In [ ]:
# !pip install -U numpy pandas scipy scikit-learn statsmodels matplotlib joblib doubleml

In [ ]:
import os
import json
import math
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import brentq
from sklearn.base import clone
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

RANDOM_STATE = 20260321
np.random.seed(RANDOM_STATE)

In [ ]:
import os
import json
import math
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import brentq
from sklearn.base import clone
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

RANDOM_STATE = 20260321
np.random.seed(RANDOM_STATE)

In [ ]:
# DoubleML 예시와 맞춘 기본 변수
Y_COL = "net_tfa"
D_COL = "e401"   # eligibility intent-to-treat
X_COLS = ["age", "inc", "educ", "fsize", "marr", "twoearn", "db", "pira", "hown"]

df = df_401k[[Y_COL, D_COL] + X_COLS].copy().dropna().reset_index(drop=True)

print("n =", len(df))
print(df.describe(include="all").T)

In [ ]:
def winsorize_series(s: pd.Series, lower_q: float, upper_q: float) -> pd.Series:
    lo, hi = s.quantile(lower_q), s.quantile(upper_q)
    return s.clip(lower=lo, upper=hi)

def winsorize_dataframe(df_in: pd.DataFrame, cols: List[str], lower_q: float, upper_q: float) -> pd.DataFrame:
    out = df_in.copy()
    for c in cols:
        out[c] = winsorize_series(out[c], lower_q, upper_q)
    return out

def make_y_learner():
    # 연속형 outcome 회귀
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LassoCV(cv=5, random_state=RANDOM_STATE, n_alphas=50, max_iter=20000))
    ])

def make_d_learner():
    # binary treatment propensity
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegressionCV(
            cv=5,
            penalty="l2",
            solver="lbfgs",
            random_state=RANDOM_STATE,
            max_iter=5000,
            scoring="neg_log_loss"
        ))
    ])

def top_share(values: np.ndarray, alpha: float) -> float:
    vals = np.abs(np.asarray(values))
    total = vals.sum()
    if total <= 0:
        return np.nan
    k = max(1, int(np.floor(alpha * len(vals))))
    topk = np.sort(vals)[::-1][:k].sum()
    return float(topk / total)

def score_ess(values: np.ndarray) -> float:
    vals = np.asarray(values)
    num = np.abs(vals).sum() ** 2
    den = np.square(vals).sum()
    if den <= 0:
        return np.nan
    return float(num / den)

def clipped(z: np.ndarray, tau: float) -> np.ndarray:
    return np.sign(z) * np.minimum(np.abs(z), tau)

def normal_ci(theta: float, se: float, alpha: float = 0.05) -> Tuple[float, float]:
    z = 1.959963984540054
    return theta - z * se, theta + z * se

def weighted_quantile(x: np.ndarray, q: float) -> float:
    return float(np.quantile(np.asarray(x), q))

In [ ]:
def crossfit_nuisance(
    df_in: pd.DataFrame,
    y_col: str,
    d_col: str,
    x_cols: List[str],
    n_splits: int = 5,
    random_state: int = RANDOM_STATE,
):
    y = df_in[y_col].to_numpy()
    d = df_in[d_col].to_numpy()
    X = df_in[x_cols].to_numpy()

    g_hat = np.zeros(len(df_in), dtype=float)  # E[Y|X]
    m_hat = np.zeros(len(df_in), dtype=float)  # E[D|X]

    if set(np.unique(d)).issubset({0, 1}):
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_iter = splitter.split(X, d)
    else:
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        split_iter = splitter.split(X)

    for train_idx, test_idx in split_iter:
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr = y[train_idx]
        d_tr = d[train_idx]

        g_model = clone(make_y_learner())
        m_model = clone(make_d_learner())

        g_model.fit(X_tr, y_tr)
        m_model.fit(X_tr, d_tr)

        g_hat[test_idx] = g_model.predict(X_te)
        m_hat[test_idx] = m_model.predict_proba(X_te)[:, 1]

    U_hat = y - g_hat
    V_hat = d - m_hat

    out = df_in.copy()
    out["_g_hat"] = g_hat
    out["_m_hat"] = m_hat
    out["_U_hat"] = U_hat
    out["_V_hat"] = V_hat
    return out

In [ ]:
@dataclass
class DMLResult:
    estimator: str
    theta: float
    se: float
    ci_low: float
    ci_high: float
    J_hat: float
    score_sd: float
    top1_share: float
    top05_share: float
    ess: float
    tau: Optional[float] = None
    q: Optional[float] = None
    bias_proxy: Optional[float] = None
    notes: Optional[str] = None

def dml_standard_from_residuals(U_hat: np.ndarray, V_hat: np.ndarray, estimator_name: str = "standard_dml") -> DMLResult:
    theta = float(np.mean(V_hat * U_hat) / np.mean(V_hat ** 2))
    psi = V_hat * (U_hat - theta * V_hat)

    J_hat = -float(np.mean(V_hat ** 2))
    infl = -psi / J_hat
    se = float(np.std(infl, ddof=1) / np.sqrt(len(infl)))
    ci_low, ci_high = normal_ci(theta, se)

    return DMLResult(
        estimator=estimator_name,
        theta=theta,
        se=se,
        ci_low=ci_low,
        ci_high=ci_high,
        J_hat=J_hat,
        score_sd=float(np.std(psi, ddof=1)),
        top1_share=top_share(psi, 0.01),
        top05_share=top_share(psi, 0.005),
        ess=score_ess(psi),
        tau=None,
        q=None,
        bias_proxy=None,
        notes="Standard orthogonal-score DML."
    )

def clipped_moment(theta: float, U_hat: np.ndarray, V_hat: np.ndarray, tau: float) -> float:
    psi = V_hat * (U_hat - theta * V_hat)
    return float(np.mean(clipped(psi, tau)))

def solve_clipped_theta(U_hat: np.ndarray, V_hat: np.ndarray, tau: float) -> float:
    # 우선 넓은 bracket 시도
    theta_std = float(np.mean(V_hat * U_hat) / np.mean(V_hat ** 2))
    grid = np.linspace(theta_std - 50000, theta_std + 50000, 401)
    vals = np.array([clipped_moment(t, U_hat, V_hat, tau) for t in grid])

    sign_change_idx = None
    for i in range(len(grid) - 1):
        if vals[i] == 0:
            return float(grid[i])
        if vals[i] * vals[i + 1] < 0:
            sign_change_idx = i
            break

    if sign_change_idx is not None:
        a, b = grid[sign_change_idx], grid[sign_change_idx + 1]
        return float(brentq(lambda t: clipped_moment(t, U_hat, V_hat, tau), a, b, maxiter=1000))

    # sign change 없으면 |moment| 최소점 사용
    idx = int(np.argmin(np.abs(vals)))
    return float(grid[idx])

def numerical_jacobian(theta: float, U_hat: np.ndarray, V_hat: np.ndarray, tau: float, h: float = 1e-4) -> float:
    f_plus = clipped_moment(theta + h, U_hat, V_hat, tau)
    f_minus = clipped_moment(theta - h, U_hat, V_hat, tau)
    return float((f_plus - f_minus) / (2 * h))

def dml_clipped_from_residuals(U_hat: np.ndarray, V_hat: np.ndarray, q: float, theta_prelim: Optional[float] = None) -> DMLResult:
    if theta_prelim is None:
        theta_prelim = float(np.mean(V_hat * U_hat) / np.mean(V_hat ** 2))

    psi_prelim = V_hat * (U_hat - theta_prelim * V_hat)
    tau = weighted_quantile(np.abs(psi_prelim), q)

    theta = solve_clipped_theta(U_hat, V_hat, tau)
    psi = V_hat * (U_hat - theta * V_hat)
    psi_clip = clipped(psi, tau)

    J_hat = numerical_jacobian(theta, U_hat, V_hat, tau)
    if abs(J_hat) < 1e-8:
        J_hat = -float(np.mean(V_hat ** 2))

    infl = -psi_clip / J_hat
    se = float(np.std(infl, ddof=1) / np.sqrt(len(infl)))
    ci_low, ci_high = normal_ci(theta, se)

    bias_proxy = float(np.mean(np.maximum(np.abs(psi_prelim) - tau, 0.0)))

    return DMLResult(
        estimator=f"score_clipped_dml_q{q}",
        theta=theta,
        se=se,
        ci_low=ci_low,
        ci_high=ci_high,
        J_hat=J_hat,
        score_sd=float(np.std(psi_clip, ddof=1)),
        top1_share=top_share(psi_clip, 0.01),
        top05_share=top_share(psi_clip, 0.005),
        ess=score_ess(psi_clip),
        tau=float(tau),
        q=float(q),
        bias_proxy=bias_proxy,
        notes="Clipped-score DML using empirical preliminary-score quantile."
    )

In [ ]:
def multiplier_bootstrap_ci(
    theta_hat: float,
    psi_values: np.ndarray,
    J_hat: float,
    B: int = 999,
    alpha: float = 0.05,
    random_state: int = RANDOM_STATE
):
    rng = np.random.default_rng(random_state)
    n = len(psi_values)
    draws = np.empty(B)

    for b in range(B):
        xi = rng.standard_normal(n)
        boot_shift = -np.mean(xi * psi_values) / J_hat
        draws[b] = theta_hat + boot_shift

    low = float(np.quantile(draws, alpha / 2))
    high = float(np.quantile(draws, 1 - alpha / 2))
    return low, high, draws

def self_normalized_ci(theta_hat: float, psi_values: np.ndarray, J_hat: float, alpha: float = 0.05):
    n = len(psi_values)
    infl = -psi_values / J_hat
    se_sn = float(np.sqrt(np.mean((infl - infl.mean()) ** 2) / n))
    low, high = normal_ci(theta_hat, se_sn, alpha=alpha)
    return low, high, se_sn

In [ ]:
def run_empirical_pipeline(
    df_in: pd.DataFrame,
    y_col: str,
    d_col: str,
    x_cols: List[str],
    clip_grid: List[float] = [0.95, 0.97, 0.99, 0.995, 0.999],
    winsor_q: Tuple[float, float] = (0.01, 0.99),
):
    # 1) standard
    cf_std = crossfit_nuisance(df_in, y_col, d_col, x_cols)
    std_res = dml_standard_from_residuals(cf_std["_U_hat"].to_numpy(), cf_std["_V_hat"].to_numpy(), "standard_dml")

    # 2) outcome winsorized
    df_yw = df_in.copy()
    df_yw[y_col] = winsorize_series(df_yw[y_col], winsor_q[0], winsor_q[1])
    cf_yw = crossfit_nuisance(df_yw, y_col, d_col, x_cols)
    yw_res = dml_standard_from_residuals(cf_yw["_U_hat"].to_numpy(), cf_yw["_V_hat"].to_numpy(), "outcome_winsorized_dml")

    # 3) covariate winsorized
    df_xw = winsorize_dataframe(df_in, x_cols, winsor_q[0], winsor_q[1])
    cf_xw = crossfit_nuisance(df_xw, y_col, d_col, x_cols)
    xw_res = dml_standard_from_residuals(cf_xw["_U_hat"].to_numpy(), cf_xw["_V_hat"].to_numpy(), "covariate_winsorized_dml")

    # 4) clipped path
    U_std = cf_std["_U_hat"].to_numpy()
    V_std = cf_std["_V_hat"].to_numpy()
    clip_results = []
    for q in clip_grid:
        clip_results.append(dml_clipped_from_residuals(U_std, V_std, q=q, theta_prelim=std_res.theta))

    # best clipped by concentration-constrained then RMSE-like proxy not available in real data.
    # empirical choice: smallest top1 share among acceptable bias proxies, fallback to lowest top1 share
    clip_df = pd.DataFrame([asdict(r) for r in clip_results])
    best_idx = clip_df["top1_share"].idxmin()
    best_clip = clip_results[int(best_idx)]

    # inference extras for standard and best clipped
    psi_std = V_std * (U_std - std_res.theta * V_std)
    std_mb_low, std_mb_high, _ = multiplier_bootstrap_ci(std_res.theta, psi_std, std_res.J_hat)
    std_sn_low, std_sn_high, std_sn_se = self_normalized_ci(std_res.theta, psi_std, std_res.J_hat)

    psi_best = V_std * (U_std - best_clip.theta * V_std)
    psi_best_clip = clipped(psi_best, best_clip.tau)
    best_mb_low, best_mb_high, _ = multiplier_bootstrap_ci(best_clip.theta, psi_best_clip, best_clip.J_hat)
    best_sn_low, best_sn_high, best_sn_se = self_normalized_ci(best_clip.theta, psi_best_clip, best_clip.J_hat)

    extra_inference = pd.DataFrame([
        {
            "estimator": "standard_dml",
            "theta": std_res.theta,
            "wald_low": std_res.ci_low,
            "wald_high": std_res.ci_high,
            "multiplier_low": std_mb_low,
            "multiplier_high": std_mb_high,
            "selfnorm_low": std_sn_low,
            "selfnorm_high": std_sn_high,
            "selfnorm_se": std_sn_se,
        },
        {
            "estimator": best_clip.estimator,
            "theta": best_clip.theta,
            "wald_low": best_clip.ci_low,
            "wald_high": best_clip.ci_high,
            "multiplier_low": best_mb_low,
            "multiplier_high": best_mb_high,
            "selfnorm_low": best_sn_low,
            "selfnorm_high": best_sn_high,
            "selfnorm_se": best_sn_se,
        }
    ])

    all_main = pd.DataFrame([asdict(std_res), asdict(yw_res), asdict(xw_res)] + [asdict(r) for r in clip_results])

    return {
        "crossfit_standard": cf_std,
        "main_results": all_main,
        "clipped_path": clip_df,
        "extra_inference": extra_inference,
        "best_clipped_result": asdict(best_clip),
    }

In [ ]:
empirical_out = run_empirical_pipeline(
    df_in=df,
    y_col=Y_COL,
    d_col=D_COL,
    x_cols=X_COLS,
    clip_grid=[0.95, 0.97, 0.99, 0.995, 0.999],
    winsor_q=(0.01, 0.99),
)

main_results = empirical_out["main_results"]
clipped_path = empirical_out["clipped_path"]
extra_inference = empirical_out["extra_inference"]

print("=== Main Results ===")
display(main_results[[
    "estimator", "theta", "se", "ci_low", "ci_high",
    "top1_share", "top05_share", "ess", "tau", "q", "bias_proxy"
]])

print("\n=== Robust Inference Comparison ===")
display(extra_inference)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. theta path
axes[0].plot(clipped_path["q"], clipped_path["theta"], marker="o")
axes[0].axhline(
    main_results.loc[main_results["estimator"] == "standard_dml", "theta"].iloc[0],
    linestyle="--"
)
axes[0].set_title("401(k): Estimate Path by Clipping Quantile")
axes[0].set_xlabel("q")
axes[0].set_ylabel("theta_hat")

# 2. top1 share
axes[1].plot(clipped_path["q"], clipped_path["top1_share"], marker="o")
axes[1].axhline(
    main_results.loc[main_results["estimator"] == "standard_dml", "top1_share"].iloc[0],
    linestyle="--"
)
axes[1].set_title("401(k): Top 1% Score Share")
axes[1].set_xlabel("q")
axes[1].set_ylabel("Top 1% share")

# 3. ESS
axes[2].plot(clipped_path["q"], clipped_path["ess"], marker="o")
axes[2].axhline(
    main_results.loc[main_results["estimator"] == "standard_dml", "ess"].iloc[0],
    linestyle="--"
)
axes[2].set_title("401(k): Score-based ESS")
axes[2].set_xlabel("q")
axes[2].set_ylabel("ESS proxy")

plt.tight_layout()
plt.show()

In [ ]:
def make_latex_table_401k(main_results: pd.DataFrame, best_estimator_name: Optional[str] = None):
    if best_estimator_name is None:
        clipped_only = main_results[main_results["estimator"].str.contains("score_clipped")]
        best_estimator_name = clipped_only.sort_values("top1_share").iloc[0]["estimator"]

    keep = main_results[main_results["estimator"].isin(["standard_dml", best_estimator_name])].copy()
    keep["95% CI"] = keep.apply(lambda r: f"[{r['ci_low']:.2f}, {r['ci_high']:.2f}]", axis=1)
    keep["Estimator"] = keep["estimator"].replace({
        "standard_dml": "Standard DML",
        best_estimator_name: f"Best clipped DML ({best_estimator_name.split('_q')[-1]})"
    })

    table_df = keep[["Estimator", "theta", "se", "95% CI", "top1_share", "top05_share", "ess"]].copy()
    table_df.columns = ["Estimator", "$\\hat\\theta$", "SE", "95\\% CI", "Top-1\\% Share", "Top-0.5\\% Share", "ESS Proxy"]
    return table_df

table_401k = make_latex_table_401k(main_results)
display(table_401k)

print(table_401k.to_latex(index=False, escape=False))

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def draw_error(dist: str, n: int, rng: np.random.Generator):
    if dist == "normal":
        return rng.normal(size=n)
    elif dist == "t5":
        return rng.standard_t(df=5, size=n)
    elif dist == "t3":
        return rng.standard_t(df=3, size=n)
    else:
        raise ValueError(f"Unknown dist: {dist}")

def simulate_plr_data(
    n: int = 2000,
    p: int = 10,
    theta0: float = 0.5,
    tail: str = "normal",
    contamination: float = 0.0,
    weak_overlap: bool = False,
    nonlinear: bool = False,
    random_state: int = RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)

    X = rng.normal(size=(n, p))
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]

    if nonlinear:
        g0 = 0.8 * np.sin(x1) + 0.5 * (x2 ** 2) - 0.3 * x3
        m_latent = 0.8 * np.sin(x1) - 0.6 * x2 + 0.4 * x3 * x1
    else:
        g0 = 0.6 * x1 - 0.5 * x2 + 0.3 * x3
        m_latent = 0.7 * x1 - 0.5 * x2 + 0.2 * x3

    if weak_overlap:
        m_latent = 2.0 * m_latent  # propensity pushed toward 0/1

    pscore = sigmoid(m_latent)
    D = rng.binomial(1, pscore, size=n)

    U = draw_error(tail, n, rng)
    Y = theta0 * D + g0 + U

    # contamination: outcome spikes
    if contamination > 0:
        idx = rng.choice(n, size=int(n * contamination), replace=False)
        Y[idx] = Y[idx] + rng.normal(loc=20.0, scale=5.0, size=len(idx))

    cols = [f"x{i+1}" for i in range(p)]
    df_sim = pd.DataFrame(X, columns=cols)
    df_sim["D"] = D
    df_sim["Y"] = Y
    return df_sim, cols

In [ ]:
def run_single_sim(
    n: int,
    tail: str,
    contamination: float,
    weak_overlap: bool,
    nonlinear: bool,
    theta0: float = 0.5,
    clip_grid: List[float] = [0.95, 0.97, 0.99, 0.995, 0.999],
    seed: int = RANDOM_STATE,
):
    df_sim, x_cols = simulate_plr_data(
        n=n,
        tail=tail,
        contamination=contamination,
        weak_overlap=weak_overlap,
        nonlinear=nonlinear,
        theta0=theta0,
        random_state=seed,
    )

    out = run_empirical_pipeline(
        df_in=df_sim.rename(columns={"Y": "Y", "D": "D"}),
        y_col="Y",
        d_col="D",
        x_cols=x_cols,
        clip_grid=clip_grid,
        winsor_q=(0.01, 0.99),
    )

    res = out["main_results"].copy()
    res["true_theta"] = theta0
    res["bias"] = res["theta"] - theta0
    res["rmse_component"] = (res["theta"] - theta0) ** 2
    res["covered"] = ((res["ci_low"] <= theta0) & (theta0 <= res["ci_high"])).astype(int)

    res["tail"] = tail
    res["contamination"] = contamination
    res["weak_overlap"] = weak_overlap
    res["nonlinear"] = nonlinear
    return res

In [ ]:
def run_simulation_grid(
    reps: int = 100,
    n: int = 2000,
    tails: List[str] = ["normal", "t5", "t3"],
    contaminations: List[float] = [0.0, 0.05],
    overlaps: List[bool] = [False, True],    # False=strong, True=weak
    nonlinear_list: List[bool] = [False, True],
):
    all_rows = []

    rep_counter = 0
    total = reps * len(tails) * len(contaminations) * len(overlaps) * len(nonlinear_list)

    for tail in tails:
        for contamination in contaminations:
            for weak_overlap in overlaps:
                for nonlinear in nonlinear_list:
                    for r in range(reps):
                        rep_counter += 1
                        seed = RANDOM_STATE + 10000 * rep_counter
                        tmp = run_single_sim(
                            n=n,
                            tail=tail,
                            contamination=contamination,
                            weak_overlap=weak_overlap,
                            nonlinear=nonlinear,
                            seed=seed,
                        )
                        tmp["rep"] = r + 1
                        all_rows.append(tmp)

                        if rep_counter % 10 == 0:
                            print(f"progress: {rep_counter}/{total}")

    sim_df = pd.concat(all_rows, axis=0, ignore_index=True)
    return sim_df

In [ ]:
# 처음에는 reps=20 정도로 테스트하고,
# 최종표는 reps=200 이상으로 늘리는 것을 권장합니다.
sim_results = run_simulation_grid(
    reps=20,
    n=1500,
    tails=["normal", "t5", "t3"],
    contaminations=[0.0, 0.05],
    overlaps=[False, True],
    nonlinear_list=[False, True],
)

print(sim_results.shape)
sim_results.head()

In [ ]:
summary_cols = ["bias", "rmse_component", "covered", "top1_share", "top05_share", "ess"]

sim_summary = (
    sim_results
    .groupby("estimator")[summary_cols]
    .agg({
        "bias": "mean",
        "rmse_component": lambda x: float(np.sqrt(np.mean(x))),
        "covered": "mean",
        "top1_share": "mean",
        "top05_share": "mean",
        "ess": "mean"
    })
    .reset_index()
    .rename(columns={
        "rmse_component": "rmse",
        "covered": "coverage"
    })
)

display(sim_summary.sort_values("rmse"))

In [ ]:
hostile = sim_results[
    (sim_results["tail"].isin(["t5", "t3"])) &
    (sim_results["contamination"] == 0.05) &
    (sim_results["weak_overlap"] == True) &
    (sim_results["nonlinear"] == True)
].copy()

hostile_summary = (
    hostile
    .groupby(["tail", "estimator"])[["bias", "rmse_component", "covered", "top1_share", "top05_share", "ess"]]
    .agg({
        "bias": "mean",
        "rmse_component": lambda x: float(np.sqrt(np.mean(x))),
        "covered": "mean",
        "top1_share": "mean",
        "top05_share": "mean",
        "ess": "mean"
    })
    .reset_index()
    .rename(columns={"rmse_component": "rmse", "covered": "coverage"})
    .sort_values(["tail", "rmse"])
)

display(hostile_summary)

In [ ]:
# severe design에서 clipped q별 path
hostile_clip = hostile[hostile["estimator"].str.contains("score_clipped")].copy()
hostile_clip["q"] = hostile_clip["q"].astype(float)

for metric in ["rmse_component", "covered", "top1_share"]:
    plt.figure(figsize=(7, 5))
    for tail in ["t5", "t3"]:
        tmp = hostile_clip[hostile_clip["tail"] == tail].groupby("q")[metric].mean().reset_index()
        if metric == "rmse_component":
            y = np.sqrt(tmp[metric].to_numpy())
            ylabel = "RMSE"
        elif metric == "covered":
            y = tmp[metric].to_numpy()
            ylabel = "Coverage"
        else:
            y = tmp[metric].to_numpy()
            ylabel = "Top 1% share"

        plt.plot(tmp["q"], y, marker="o", label=tail)

    plt.xlabel("Clipping quantile q")
    plt.ylabel(ylabel)
    plt.title(f"Hostile design: {ylabel} by clipping threshold")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
os.makedirs("results_dml_heavytail", exist_ok=True)

main_results.to_csv("results_dml_heavytail/401k_main_results.csv", index=False)
clipped_path.to_csv("results_dml_heavytail/401k_clipped_path.csv", index=False)
extra_inference.to_csv("results_dml_heavytail/401k_inference_compare.csv", index=False)

sim_results.to_csv("results_dml_heavytail/simulation_raw_results.csv", index=False)
sim_summary.to_csv("results_dml_heavytail/simulation_summary.csv", index=False)
hostile_summary.to_csv("results_dml_heavytail/simulation_hostile_summary.csv", index=False)

bundle = {
    "dataset_info": {
        "dataset": "401K via doubleml.datasets.fetch_401K",
        "n_obs": int(len(df)),
        "y_col": Y_COL,
        "d_col": D_COL,
        "x_cols": X_COLS,
    },
    "best_clipped_result": empirical_out["best_clipped_result"],
    "main_results_records": main_results.to_dict(orient="records"),
    "inference_records": extra_inference.to_dict(orient="records"),
    "simulation_summary_records": sim_summary.to_dict(orient="records"),
    "hostile_summary_records": hostile_summary.to_dict(orient="records"),
}

with open("results_dml_heavytail/results_bundle.json", "w", encoding="utf-8") as f:
    json.dump(bundle, f, ensure_ascii=False, indent=2)

print("Saved to results_dml_heavytail/")

In [ ]:
def compact_table(df_in: pd.DataFrame, cols: List[str], digits: int = 4) -> str:
    tmp = df_in[cols].copy()
    for c in tmp.columns:
        if pd.api.types.is_numeric_dtype(tmp[c]):
            tmp[c] = tmp[c].map(lambda x: round(float(x), digits))
    return tmp.to_markdown(index=False)

best_clip_name = main_results[main_results["estimator"].str.contains("score_clipped")]\
    .sort_values("top1_share").iloc[0]["estimator"]

report = {
    "401k_main_results_markdown": compact_table(
        main_results[main_results["estimator"].isin(["standard_dml", "outcome_winsorized_dml", "covariate_winsorized_dml", best_clip_name])],
        ["estimator", "theta", "se", "ci_low", "ci_high", "top1_share", "top05_share", "ess", "q", "tau", "bias_proxy"]
    ),
    "401k_clipped_path_markdown": compact_table(
        clipped_path.sort_values("q"),
        ["estimator", "q", "theta", "se", "top1_share", "top05_share", "ess", "bias_proxy"]
    ),
    "inference_compare_markdown": compact_table(
        extra_inference,
        ["estimator", "theta", "wald_low", "wald_high", "multiplier_low", "multiplier_high", "selfnorm_low", "selfnorm_high", "selfnorm_se"]
    ),
    "simulation_summary_markdown": compact_table(
        sim_summary.sort_values("rmse"),
        ["estimator", "bias", "rmse", "coverage", "top1_share", "top05_share", "ess"]
    ),
    "hostile_summary_markdown": compact_table(
        hostile_summary.sort_values(["tail", "rmse"]),
        ["tail", "estimator", "bias", "rmse", "coverage", "top1_share", "top05_share", "ess"]
    )
}

print("=" * 100)
print("아래 내용을 그대로 ChatGPT에게 보내면 후속 개정/추가 코드 요청에 바로 쓸 수 있습니다.")
print("=" * 100)

for k, v in report.items():
    print(f"\n\n## {k}\n")
    print(v)

print("\n\n추가로 같이 보내면 좋은 파일:")
print("- results_dml_heavytail/results_bundle.json")
print("- results_dml_heavytail/401k_main_results.csv")
print("- results_dml_heavytail/401k_clipped_path.csv")
print("- results_dml_heavytail/simulation_summary.csv")
print("- results_dml_heavytail/simulation_hostile_summary.csv")